# 0. MCP Server Setup (Optional)

*Estimated time to run notebook: about 12-15 min*

**Goal:** Create and deploy a simple DataRobot MCP server once, then reuse it across the notebooks.

**Context:** [Model Context Protocol (MCP)](https://docs.datarobot.com/en/docs/agentic-ai/agentic-mcp/agentic-tools-mcp.html#integrate-tools-using-an-mcp-server) lets an agent call tools backed by a DataRobot deployment. In this workshop, notebooks 3–5 use `MCP_DEPLOYMENT_ID` so your agent can reach DataRobot through that server’s tools.

**Outcome:** When you finish this notebook, you will have deployed an MCP server (or chosen an existing one) and set `MCP_DEPLOYMENT_ID` in the workshop `.env` (repository root) so notebooks 3–5 can use it.

**Reference:** [datarobot-mcp-template](https://github.com/datarobot-community/datarobot-mcp-template) (official template repo)

**Run the following cell** to load `MCP_STACK_NAME` from `.env`.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Workshop repo root (whether the kernel cwd is here or inside `datarobot-mcp-template/` after Step 1)
p = Path.cwd()
WORKSHOP_ROOT = p.parent if p.name == "datarobot-mcp-template" else p

load_dotenv(WORKSHOP_ROOT / ".env", override=True)
MCP_STACK_NAME = os.getenv("MCP_STACK_NAME", "").strip()
if not MCP_STACK_NAME:
    raise ValueError(
        "Set MCP_STACK_NAME in workshop .env (see .env.example), e.g. my-mcp-stack-your-initials"
)

## Step 1: Prerequisites and clone

**Where to run from:** Start in the **workshop repository root**—the directory that contains this notebook, `README.md`, `pyproject.toml`, and `uv.lock`. The clone step uses `%cd` to move the notebook kernel into `datarobot-mcp-template` for later cells (until you restart the kernel).

The next cells install or assume:

- **Python** (3.11+), **Task** ([Taskfile](https://taskfile.dev/) CLI), **uv**, and **Pulumi**. The template’s `task install` wires up the project.
- **DataRobot credentials** for `dr_mcp/.env` and for deployment.
- A **clone** of the MCP template in a `datarobot-mcp-template` subfolder (the next cell creates it if missing).

In [ ]:
# Install workshop Python deps from the checked-in uv.lock
import sys
!pip install "pydantic-ai==1.0.18" -q

In [ ]:
# Run from workshop repo root (folder containing uv.lock). This cell switches cwd to the clone.
# Clone official DataRobot MCP template (README quickstart) — stored in a dedicated subfolder
!test -d datarobot-mcp-template || git clone https://github.com/datarobot-community/datarobot-mcp-template.git
%cd datarobot-mcp-template

# Install project dependencies
!task --silent install >/dev/null 2>&1

# Create env file for MCP app config
!test -f dr_mcp/.env || cp .env.template dr_mcp/.env

print("MCP server repository cloned in dedicated datarobot-mcp-template subfolder.")

## Step 2: Deploy the MCP server in DataRobot (or reuse an existing one)

**Needs:** `dr_mcp/.env` filled for deploy (API token, endpoint, and template secrets). `MCP_STACK_NAME` must already be loaded from your workshop `.env`.

The next cell runs `task infra:init` then `task infra:up-yes` (non-interactive). If the stack exists already, `infra:init` fails—then run only `task infra:up-yes` after `cp dr_mcp/.env .env`, or drop the conflicting stack in Pulumi.


In [ ]:
import subprocess, os
from pathlib import Path

def stream(cmd, cwd=None):
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    return proc.returncode

work_dir = WORKSHOP_ROOT / "datarobot-mcp-template"

# Step 1: copy .env
stream("cp dr_mcp/.env .env", cwd=work_dir)
print("✅ .env copied")

# Step 2: init or select stack
rc = stream(f"task infra:init -- {MCP_STACK_NAME}", cwd=work_dir)
if rc != 0:
    print("Stack exists, selecting...")
    stream(f"task infra:select -- {MCP_STACK_NAME}", cwd=work_dir)
print("✅ stack ready")

# Step 3: deploy
stream("task infra:up-yes", cwd=work_dir)
print("✅ deployed")

**Set `MCP_DEPLOYMENT_ID`** in your workshop `.env` after deploy: use **Deployments** in the DataRobot app (deployment overview or URL: `/deployments/<id>/`), or the **Deployment Id** in Pulumi output. Skip deploy if you reuse an existing deployment.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(WORKSHOP_ROOT / ".env", override=True)
mcp_deployment_id = os.getenv("MCP_DEPLOYMENT_ID", "").strip()

if not mcp_deployment_id:
    print("MCP_DEPLOYMENT_ID not set yet—add it to workshop .env after deploy (or skip this notebook).")
else:
    print(f"MCP_DEPLOYMENT_ID is set: {mcp_deployment_id} (continue to notebooks 1–5).")

⚠️ Note on Kernel Limits: DataRobot Codespaces have a limit of 5 active notebook kernels at a time. To ensure a smooth transition to the next exercise, please remember to shut down this kernel (by closing the notebook tab) once you are finished. This prevents any 'limit reached' errors when opening subsequent notebooks!